In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp,current_timestamp

In [0]:
dbutils.widgets.text("bronze_catalog","dbr_dev")
dbutils.widgets.text( "bronze_schema","artemzharkov10_bronze")

dbutils.widgets.text("silver_catalog","dbr_dev")
dbutils.widgets.text("silver_schema","artemzharkov10_silver")

BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
    

In [0]:

# print(f"Bronze row count: {df_bronze.count()}")

In [0]:
df_bronze = spark.read.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.gdelt_history_bronze")
df_clean = df_bronze.dropna(subset=["SqlDate","GlobalEventID","SourceUrl","Actor1Name"])
df_remove_dublicate = df_clean.dropDuplicates(["GlobalEventID"])
df_silver = (
  df_remove_dublicate
    .withColumn("NewsDateClean", to_date(col("SqlDate").cast("string"), "yyyyMMdd"))
    .withColumn("DateAddedClean", to_date(col("DateAdded").cast("string"), "yyyyMMdd"))
    .withColumn("GlobalEventID", col("GlobalEventID").cast("long"))
    .withColumn("QuadClass", col("QuadClass").cast("integer"))
    .withColumn("GoldsteinScale", col("GoldsteinScale").cast("double"))
    .withColumn("NumMentions", col("NumMentions").cast("integer"))
    .withColumn("NumSources", col("NumSources").cast("integer"))
    .withColumn("NumArticles", col("NumArticles").cast("integer"))
    .withColumn("AvgTone", col("AvgTone").cast("double"))
    
    .withColumn("ActionGeo_Lat", col("ActionGeo_Lat").cast("double"))
    .withColumn("ActionGeo_Long", col("ActionGeo_Long").cast("double"))
    #meta
    .withColumn("source_filename", col("_metadata.file_path"))
    .withColumn("ingest_timestamp", current_timestamp())
  )

(df_silver.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.gdelt_history_silver"))

In [0]:
# b = df_bronze.count()
# s = df_silver.count()
# print(f"Bronze records: {b}")
# print(f"Silver records: {s}")
# print(f"Records is filtered (removed): {b-s}")